# Standard RAG pipeline

In [1]:
import os
import time
from typing import List, Dict, Any
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from transformers import pipeline

c:\Users\arion\rag-classification\standard_rag\RAG_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import transformers
import accelerate
import huggingface_hub
import sentence_transformers


print(transformers.__version__)
print(accelerate.__version__)
print(huggingface_hub.__version__)
print(sentence_transformers.__version__)



4.39.3
1.12.0
0.36.2
2.7.0


In [4]:
from rag_notebooks_data.test import *

## 1 Preprocessing of jupyter json test files

In [ ]:
import json

def extract_code_cells_from_notebook(notebook_path):
    """To preprocess the input of the RAG, not the corpus documents

    Args:
        notebook_path (string): raw sting of the path of the notebook

    Raises:
        FileNotFoundError: _description_
        ValueError: _description_

    Returns:
        _type_: preproccesed notebook
    """
    try:
        with open(notebook_path, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"Notebook file not found: {notebook_path}")
    except json.JSONDecodeError:
        raise ValueError(f"Notebook file is not valid JSON: {notebook_path}")

    code_blocks = []
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') == 'code':
            code = ''.join(cell.get('source', []))
            code_blocks.append(code)
    return code_blocks


In [ ]:
#test1 = extract_code_cells_from_notebook(r"rag_notebooks_data\test\Spotify_DL.json")
#test2 = extract_code_cells_from_notebook(r"rag_notebooks_data\test\Spotify_ML.json")
test3 = extract_code_cells_from_notebook(r"rag_notebooks_data\test\notebook2.json")

In [7]:
test1[8]


"train = train[train['track_popularity'] != 0 ]\ntrain = train[ train['acousticness'] != 0]"

## 2 Vector database setup

In [8]:
def load_and_chunk_documents():
    """
    Load sample notebooks example classfication and chunk them for better retrieval.
    
    """
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    policy_documents = [
        {
            "desc": "Classic ML fit (should detect Model Training)",
        "code": """ 
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)"""
        },
        {
             "desc": "Data loading (should detect Data Loading)",
        "code": """
import pandas as pd
df = pd.read_csv('data.csv')
"""
        },
        {
             "desc": "Model evaluation (should detect Model Evaluation)",
        "code": """
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred))
"""
        },
        {
            "desc": "Deep learning training (should detect Model Training, Deep Learning)",
        "code": """
import tensorflow as tf
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(10))
model.compile(optimizer='adam', loss='mse')
model.fit(X, y, epochs=5)
"""
        },
        {
            "desc": "Empty code (should return nothing or fallback)",
        "code": ""
        },
        {
            "desc": "Non-ML code (should not hallucinate ML steps)",
        "code": """
print("Hello, world!")
a = 5 + 3
"""
        },
        {
            "desc": "Multiple ML steps in one cell (should detect all present steps)",
        "code": """
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
X_train, X_test, y_train, y_test = train_test_split(X, y)
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
"""
        },
        {
            "desc": "Unusual ML library (should fallback or try to detect)",
        "code": """
import xgboost as xgb
model = xgb.XGBClassifier()
model.fit(X_train, y_train)
"""
        },
        {
            "desc": "Malformed code (should not crash, should fallback)",
        "code": """
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)
"""
        },
        {
            "desc": "Comments only (should fallback)",
        "code": """
# This cell only contains comments
# No executable code here"""
        },
        {
            "desc": "Irrelevant imports (should not hallucinate ML steps)",
        "code": """
import os
import sys
print(os.getcwd())
"""
        },
        {
            "desc": "Ambiguous variable names (should not guess)",
        "code": """
foo = bar(X, y)
baz = foo.predict(X)
"""

        },
        {
            "desc": "Only function definitions (should fallback)",
        "code": """
def my_function(x):
    return x * 2
"""

        },
        {
            "desc": "ML and non-ML mixed (should only detect ML parts)",
        "code": """
import pandas as pd
df = pd.read_csv('data.csv')
print("Loaded data")
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(df['X'], df['y'])
print("Done")
"""

        }
    ]
    
    print(f"Loaded notebook code examples classification successful")
    
    # Configure text splitter 
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", " ", ""]
    )
    
    # Chunk all documents
    all_chunks = []
    for doc in policy_documents:
        chunks = text_splitter.split_text(doc["code"])
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "desc": doc["desc"],
                "code": chunk,
            })
    
    return all_chunks

In [17]:
def setup_vector_database(chunks):
    """
    Set up ChromaDB vector database and store document chunks.
    
    """
    import chromadb
    """ 
    
    try:
        client.delete_collection(name="notebook_examples_policy")
    except:
        pass
    """
    # Initialize ChromaDB client
    client = chromadb.Client()
    
    # Create collection or get it
    
    collection = client.get_or_create_collection(name="notebook_examples_policy")#  collection name with underscores and no spaces
    
    
    # Prepare data for storage
    ids = [f"chunk_{i}" for i in range(len(chunks))]
    codes = [chunk["code"] for chunk in chunks]
    descriptions = [{"desc": chunk["desc"]} for chunk in chunks]
    
    # Adding documents to collection & embedding
    collection.add(
        ids=ids,
        documents=codes,
        metadatas=descriptions
    )
        
    print("Vector Database Created")
    return collection

## 3. Query processing

In [10]:
def process_user_query(query: str):
    """
    Process user query and convert to embedding for vector search.
    
    """
    from sentence_transformers import SentenceTransformer
    
    # Loading embedding model 
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Preprocess query
    cleaned_query = str(query).lower().strip()
    
    # Convert query to embedding
    query_embedding = model.encode(cleaned_query)

    return query_embedding

## 4. Vector search

In [11]:
def search_vector_database(collection, query_embedding, top_k: int = 3):
    """
    Search vector database for relevant document chunks.
    """
    
    # Performing vector search
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )
    
    # Process results
    search_results = []
    for i in range(len(results['ids'][0])):
        doc_id = results['ids'][0][i]
        distance = results['distances'][0][i]
        document = results['documents'][0][i]
        metadata = results['metadatas'][0][i]
        
        similarity = 1 - distance
        search_results.append({
            'id': doc_id,
            'content': document,
            'metadata': metadata,
            'similarity': similarity
        })
    
    return search_results

## 5. Context augmentation

In [12]:
def augment_prompt_with_context(query: str, search_results: List[Dict]) -> str:
    """
    Build augmented prompt with retrieved context for LLM.

    """
    # Assemble context from search results
    context_parts = []
    for i, result in enumerate(search_results, 1):
        context_parts.append(f"Source {i}: {result['metadata']}\n{result['content']}")
    
    context = "\n\n".join(context_parts[:5]) #max 5 chunks
    
    
    # Build augmented prompt
    augmented_prompt = f"""
You are an assistant. Answer using only the context, and based on the following notebook chunk code classification examples, classify all the code chunks on this notebook.

Context:
{context}

Given notebook:
{query}

Answer:
"""
    
    return augmented_prompt

## 6. Response generation

In [24]:
# Loading the LLM model
from transformers import pipeline
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    device = -1
)

In [25]:
def generate_response(augmented_prompt):
    
    """Generating an answer for the classification task by using an LLM

    Returns:
        _type_: asnwer from the LLM (string)
    """
    from transformers import pipeline
    """ 
    generator = pipeline(
        "text-generation",
        model='mistralai/Mistral-7B-Instruct-v0.2'  # or 'google/flan-t5-large'
    )
    """
    
    response = generator(
        augmented_prompt,
        max_new_tokens=4000,
        temperature=0.0,
        do_sample=False
    )
    
    return response[0]["generated_text"]

In [27]:
def run_complete_rag_pipeline(query):
    """
    Running the complete RAG pipeline from start to finish.
    Here the query parameter is the notebook we put in entry, for it to be classified by a LLM.

    """
    
    # Step 1: Load and chunk documents
    chunks = load_and_chunk_documents()
    
    # Step 2: Setup vector database
    collection = setup_vector_database(chunks)
    print(f"DEBUG - collection = {collection}")  
    print(f"DEBUG - type = {type(collection)}")
    # Step 3: Process user query
    query_embedding = process_user_query(query)
    print(f"DEBUG - embedding shape: {query_embedding.shape}")
    # Step 4: Search vector database
    search_results = search_vector_database(collection, query_embedding)
    
    # Step 5: Augment prompt with context
    augmented_prompt = augment_prompt_with_context(query, search_results)
    
    # Step 6: Generate response
    response = generate_response(augmented_prompt)
    
    # Display final result
    print("FINAL RESULT\n")
    
    
    return response

In [31]:
run_complete_rag_pipeline(test3)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loaded notebook code examples classification successful


Insert of existing embedding ID: chunk_0
Insert of existing embedding ID: chunk_1
Insert of existing embedding ID: chunk_2
Insert of existing embedding ID: chunk_3
Insert of existing embedding ID: chunk_4
Insert of existing embedding ID: chunk_5
Insert of existing embedding ID: chunk_6
Insert of existing embedding ID: chunk_7
Insert of existing embedding ID: chunk_8
Insert of existing embedding ID: chunk_9
Insert of existing embedding ID: chunk_10
Insert of existing embedding ID: chunk_11
Insert of existing embedding ID: chunk_12
Add of existing embedding ID: chunk_0
Add of existing embedding ID: chunk_1
Add of existing embedding ID: chunk_2
Add of existing embedding ID: chunk_3
Add of existing embedding ID: chunk_4
Add of existing embedding ID: chunk_5
Add of existing embedding ID: chunk_6
Add of existing embedding ID: chunk_7
Add of existing embedding ID: chunk_8
Add of existing embedding ID: chunk_9
Add of existing embedding ID: chunk_10
Add of existing embedding ID: chunk_11
Add of

Vector Database Created
DEBUG - collection = name='notebook_examples_policy' id=UUID('b1c44a86-c348-4af7-91a5-c8f1d8923131') metadata=None tenant='default_tenant' database='default_database'
DEBUG - type = <class 'chromadb.api.models.Collection.Collection'>


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


DEBUG - embedding shape: (384,)


This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (1024). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


FINAL RESULT



"\nYou are an assistant. Answer using only the context, and based on the following notebook chunk code classification examples, classify all the code chunks on this notebook.\n\nContext:\nSource 1: {'desc': 'Model evaluation (should detect Model Evaluation)'}\nfrom sklearn.metrics import classification_report\nprint(classification_report(y_true, y_pred))\n\nSource 2: {'desc': 'Irrelevant imports (should not hallucinate ML steps)'}\nimport os\nimport sys\nprint(os.getcwd())\n\nSource 3: {'desc': 'Classic ML fit (should detect Model Training)'}\nfrom sklearn.ensemble import RandomForestClassifier\nmodel = RandomForestClassifier()\nmodel.fit(X_train, y_train)\n\nGiven notebook:\n['import sklearn\\nimport numpy', '1+1']\n\nAnswer:\n'Model evaluation (should detect Model Evaluation)'}\nfrom sklearn.ensemble import RandomForestClassifier\nmodel = RandomForestClassifier()\nmodel.fit(X_train, y_train)\nGiven notebook:\n['import sklearn\\nimport numpy', '1+1']\nAnswer:\n'Model evaluation (shoul

In [32]:
# END

!pip freeze > requirements.txt

I want to thank HuggingFace, Joey O'Neill and Jeremy Morgan